# Zonal raster statistics

Summarize raster values by an aligned integer zone grid. This is a grouped operation, not a moving neighbourhood filter.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'implementations').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'implementations/python/zonal-statistics'))
from zonal_statistics import zonal_statistics


## 1. Define values and zones

The two arrays must be aligned cell by cell. In a real GIS workflow, verify CRS, transform, resolution and origin as well as array shape.

In [ ]:
values=np.array([[10,12,15,18],[11,13,16,19],[9,14,17,20],[8,np.nan,18,21]],dtype=float)
zones=np.array([[1,1,2,2],[1,1,2,2],[3,3,2,2],[3,3,3,2]])
fig,axes=plt.subplots(1,2,figsize=(8,4))
for ax,data,title in zip(axes,[values,zones],['Raster values','Zone labels']):
    im=ax.imshow(data,cmap='viridis'); ax.set_title(title); fig.colorbar(im,ax=ax)
plt.tight_layout(); plt.show()


## 2. Calculate zonal summaries

The implementation reports count, sum, mean, minimum, maximum, median and population standard deviation. Non-finite values are skipped; zones with no valid values are retained.

In [ ]:
summary=zonal_statistics(values,zones)
for zone,stats in summary.items():
    print(f'Zone {zone}: {stats}')


## 3. Distinguish NoData from an absent zone

A zone whose values are all NoData still appears in the result with count zero. A NoData zone label, by contrast, is excluded.

In [ ]:
with_empty_zone=zones.copy(); with_empty_zone[values.shape[0]-1,:]=9
with_empty_values=values.copy(); with_empty_values[-1,:]=np.nan
print(zonal_statistics(with_empty_values,with_empty_zone)[9])


## Investigate

1. Calculate one zone's mean by hand.
2. What changes when a value becomes NoData?
3. Why retain a zone with no valid values?
4. Why does matching array shape not prove geospatial alignment?
5. Explain the difference between focal and zonal statistics.